In [1]:
print(1)

1


In [2]:
print(2)

2


In [3]:
import os
import sys
from pathlib import Path

import torch
from transformers import AutoTokenizer

# This is a complete local Hugging Face snapshot on the external disk.
# `local_files_only=True` below makes a missing file fail instead of downloading it.
MODEL_DIR = Path(
    '/extra_disk_1/vasilievpavel/dlm-attn-res/huggingface/hub/'
    'models--GSAI-ML--LLaDA-8B-Base/snapshots/'
    '0f2787f2d87eac5eed8a087d5ecd24277e6255b2'
)
assert MODEL_DIR.is_dir(), f'Local checkpoint is missing: {MODEL_DIR}'
os.environ['HF_HUB_OFFLINE'] = '1'

# Make our local architecture importable whether Jupyter starts in the repo root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from dlm_attn_res.models.llada import LLaDAConfig, LLaDAModelLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
config = LLaDAConfig.from_pretrained(MODEL_DIR, local_files_only=True)
model = LLaDAModelLM.from_pretrained(
    MODEL_DIR, config=config, torch_dtype=torch.bfloat16, local_files_only=True
).cuda().eval()

/home/vasilievpavel/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 6/6 [00:04<00:00,  1.49it/s]
Some weights of LLaDAModelLM were not initialized from the model checkpoint at /extra_disk_1/vasilievpavel/dlm-attn-res/huggingface/hub/models--GSAI-ML--LLaDA-8B-Base/snapshots/0f2787f2d87eac5eed8a087d5ecd24277e6255b2 and are newly initialized: ['model.transformer.blocks.0.attn_res.pseudo_query', 'model.transformer.blocks.0.norm.weight', 'model.transformer.blocks.1.attn_res.pseudo_query', 'model.transformer.blocks.1.norm.weight', 'model.transformer.blocks.10.attn_res.pseudo_query', 'model.transformer.blocks.10.norm.weight', 'model.transformer.

In [4]:
prompt = 'What is going on?'
encoded = tokenizer(prompt, return_tensors='pt')
device = next(model.parameters()).device
input_ids = encoded['input_ids'].to(device)
attention_mask = encoded['attention_mask'].to(device)

output = model(input_ids=input_ids, attention_mask=attention_mask)

logits = output.logits  # [batch, sequence_length, vocabulary_size]
print(logits.shape)

torch.Size([1, 5, 126464])


In [5]:
import torch
import torch.nn as nn

from attn_res import AttnResOperator, RMSNorm


class MyTransformerBlock(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.attn_res = AttnResOperator(d_model)
        self.norm = RMSNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, num_heads=8, batch_first=True)

    def forward(self, sources):
        """
        sources: Tensor of shape (N_src, B, T, d_model)
            e.g. stack of [embedding, block_0, ..., block_{k-1}, partial_block]
        """
        # Depth-wise attention over all sources → input h_l
        h = self.attn_res(sources)          # (B, T, d_model)

        # Plug h into any transformer-style sub-layer
        h_norm = self.norm(h)
        attn_out, _ = self.self_attn(h_norm, h_norm, h_norm)

        # Update your own notion of block / layer outputs as usual
        return attn_out


# Example usage inside a custom stack
B, T, d_model = 2, 64, 512
embedding = torch.randn(B, T, d_model)
prev_block = torch.randn(B, T, d_model)

block = MyTransformerBlock(d_model)
sources = torch.stack([embedding, prev_block], dim=0)  # (N_src=2, B, T, d_model)
out = block(sources)
print(out.shape)  # (2, 64, 512)

torch.Size([2, 64, 512])
